In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS viajes_aereos_oro;

-- 1. Dimensión Tiempo / Calendario
CREATE OR REPLACE TABLE viajes_aereos_oro.dim_tiempo AS
SELECT DISTINCT
    CAST(DATE_FORMAT(salida_programada, 'yyyyMMdd') AS INT) AS id_tiempo,
    DATE(salida_programada) AS fecha,
    YEAR(salida_programada) AS anio,
    QUARTER(salida_programada) AS trimestre,
    MONTH(salida_programada) AS mes,
    DATE_FORMAT(salida_programada, 'MMMM') AS nombre_mes,
    DAY(salida_programada) AS dia,
    DAYOFWEEK(salida_programada) AS dia_semana
FROM viajes_aereos_plata.fact_vuelos
WHERE salida_programada IS NOT NULL;

-- 2. Tabla de Hechos Central (Star Schema - Kimball)
CREATE OR REPLACE TABLE viajes_aereos_oro.gold_fact_ventas_vuelos AS
SELECT 
    dt.numero_tiquete,
    dt.id_vuelo,
    r.codigo_reserva,
    t.id_pasajero,
    v.codigo_avion,
    v.aeropuerto_origen,
    v.aeropuerto_destino,
    CAST(DATE_FORMAT(v.salida_programada, 'yyyyMMdd') AS INT) AS id_tiempo_salida,
    dt.clase_tarifa,
    dt.monto_tarifa,
    r.monto_total_reserva,
    v.estado_vuelo
FROM viajes_aereos_plata.fact_detalle_tiquetes dt
INNER JOIN viajes_aereos_plata.fact_vuelos v ON dt.id_vuelo = v.id_vuelo
INNER JOIN viajes_aereos_plata.dim_pasajeros_tiquetes t ON dt.numero_tiquete = t.numero_tiquete
INNER JOIN viajes_aereos_plata.fact_reservas r ON t.codigo_reserva = r.codigo_reserva;

-- 3. Vista de KPIs Analíticos
CREATE OR REPLACE VIEW viajes_aereos_oro.vw_kpi_ventas_rutas AS
SELECT 
    f.aeropuerto_origen,
    f.aeropuerto_destino,
    f.clase_tarifa,
    COUNT(DISTINCT f.numero_tiquete) AS total_pasajeros,
    SUM(f.monto_tarifa) AS ingresos_totales,
    ROUND(AVG(f.monto_tarifa), 2) AS tarifa_promedio
FROM viajes_aereos_oro.gold_fact_ventas_vuelos f
GROUP BY f.aeropuerto_origen, f.aeropuerto_destino, f.clase_tarifa;

-- 4. Verificación de tablas creadas en la Capa Oro
SHOW TABLES IN viajes_aereos_oro;

database,tableName,isTemporary
viajes_aereos_oro,dim_tiempo,false
viajes_aereos_oro,gold_fact_ventas_vuelos,false
viajes_aereos_oro,vw_kpi_ventas_rutas,false
